In [ ]:
# !pip install yfinance openpyxl  # uncomment & run first if needed

import numpy as np, pandas as pd, yfinance as yf, matplotlib.pyplot as plt, seaborn as sns, warnings
from scipy.optimize import minimize
from datetime import datetime
warnings.filterwarnings('ignore')
plt.style.use('default'); sns.set_palette("husl")

class FactorScreener:
    def __init__(self, tickers, start='2019-01-01', end=None):
        self.tickers = tickers if isinstance(tickers, list) else [tickers]
        self.start, self.end = start, end or datetime.now().strftime('%Y-%m-%d')
        self.price_data, self.financial_data = None, None
    def fetch_data(self):
        try:
            px = yf.download(self.tickers, start=self.start, end=self.end, progress=False)
            if px.empty: raise ValueError('No price data')
            if isinstance(px.columns, pd.MultiIndex):
                self.price_data = px['Adj Close'] if 'Adj Close' in px.columns.get_level_values(0) else px.iloc[:, :len(self.tickers)]
            else:
                self.price_data = pd.DataFrame(px['Adj Close']) if 'Adj Close' in px.columns else pd.DataFrame(px)
            if isinstance(self.price_data, pd.Series): self.price_data = self.price_data.to_frame()
            if len(self.price_data.columns) == len(self.tickers): self.price_data.columns = self.tickers
        except Exception as e:
            print('❌ price error:', e); self.price_data = pd.DataFrame()
        fd = {}
        for t in self.tickers:
            try:
                info = yf.Ticker(t).info
                fd[t] = {k: info.get(k2, np.nan) for k, k2 in [
                    ('current_price','currentPrice'),('market_cap','marketCap'),('pe_ratio','trailingPE'),
                    ('pb_ratio','priceToBook'),('roe','returnOnEquity'),('profit_margin','profitMargins'),
                    ('revenue_growth','revenueGrowth'),('beta','beta'),('free_cash_flow','freeCashflow'),
                    ('operating_cash_flow','operatingCashflow'),('sector','sector')]}
            except: fd[t] = {k: np.nan for k in ['current_price','market_cap','pe_ratio','pb_ratio','roe','profit_margin','revenue_growth','beta','free_cash_flow','operating_cash_flow','sector']}
        self.financial_data = pd.DataFrame(fd).T
        return self.price_data, self.financial_data
    def factors(self):
        valid = self.price_data.dropna(axis=1, thresh=252).columns
        px = self.price_data[valid]; ret = px.pct_change()
        mom = (px/px.shift(252)-1 - (px/px.shift(21)-1)).iloc[-1]
        qual = pd.Series([(self.financial_data.loc[s,'roe'] or 0 + self.financial_data.loc[s,'profit_margin'] or 0)/2 for s in valid], index=valid)
        val  = pd.Series([-(self.financial_data.loc[s,'pe_ratio']+self.financial_data.loc[s,'pb_ratio']) for s in valid if pd.notna(self.financial_data.loc[s,'pe_ratio']) and pd.notna(self.financial_data.loc[s,'pb_ratio'])], index=valid)
        lowv = -ret.rolling(252).std().iloc[-1]
        size = pd.Series([-self.financial_data.loc[s,'market_cap'] for s in valid], index=valid)
        df = pd.DataFrame({'mom':mom,'qual':qual,'val':val,'lowv':lowv,'size':size}).rank(pct=True)
        df['composite'] = df.mean(axis=1, skipna=True)
        return df.sort_values('composite', ascending=False)

class DCF:
    def __init__(self, tickers, fd, yrs=5, g=0.025):
        self.tickers, self.fd, self.yrs, self.g = tickers, fd, yrs, g
        self.details = {}
    def vals(self):
        print('💰 DCF...')
        vals = {}
        for t in self.tickers:
            if t not in self.fd.index: continue
            fcf = self.fd.loc[t,'free_cash_flow']; rev = self.fd.loc[t,'operating_cash_flow'] or 1
            gr  = self.fd.loc[t,'revenue_growth'] or 0.05; beta = self.fd.loc[t,'beta'] or 1.0
            price = self.fd.loc[t,'current_price']
            if pd.isna(fcf) or fcf<=0: continue
            r = 0.045 + beta*0.06
            proj = [rev*(1+gr)**i * (fcf/rev*0.7) for i in range(1,self.yrs+1)]
            term = proj[-1]*(1+self.g)/(r-self.g)
            pvfcf = sum([proj[i]/(1+r)**(i+1) for i in range(self.yrs)])
            pvterm = term/(1+r)**self.yrs
            iv = (pvfcf+pvterm)/1e9
            self.details[t] = {'iv':iv,'mos':(iv-price)/iv if iv>0 else np.nan}
            vals[t] = iv
        return pd.DataFrame.from_dict(vals, orient='index', columns=['iv'])

class ReturnProj:
    def __init__(self, tickers, scr, dcf):
        self.tickers, self.scr, self.dcf = tickers, scr, dcf
    def proj(self):
        print('📈 Project returns...')
        fct = self.scr.factors(); prem = {'mkt':0.06,'val':0.03,'mom':0.04,'qual':0.02,'lowv':0.02,'size':0.01}
        ivs = self.dcf.vals()
        rf = 0.045; ret = {}; attr = {}
        for t in self.tickers:
            if t not in fct.index or t not in ivs.index: continue
            score = fct.loc[t,'composite']
            fcret = rf + prem['mkt'] + sum((fct.loc[t,factor]-0.5)*prem[factor]*0.2 for factor in ['val','mom','qual','lowv','size'] if factor in fct.columns)
            price = self.scr.financial_data.loc[t,'current_price']
            iv = ivs.loc[t,'iv']
            dcfret = (iv/price)**(1/5)-1 if pd.notna(price) and pd.notna(iv) and price>0 else 0
            blended = 0.7*fcret + 0.3*dcfret
            attr[t] = {'fcret':fcret,'dcfret':dcfret,'blend':blended}
            ret[t] = blended
        self.attr = pd.DataFrame(attr)
        return pd.Series(ret)

class MeanVarOpt:
    def __init__(self, er, px, c=None):
        self.er, self.px, self.c = er, px, c or {}
    def opt(self):
        print('📊 Mean-Variance Optimization...')
        stocks = self.er.dropna().index.intersection(self.px.columns)
        er, px = self.er[stocks], self.px[stocks]
        if len(stocks)<3:
            w = pd.Series(1/len(stocks), index=stocks); return {'w':w,'er':er.mean(),'vol':0.15,'sharpe':0.5}
        rets = px.pct_change().dropna(); cov = rets.cov()*252
        n = len(stocks); bnds = self.c.get('bounds',[(0,0.3)]*n)
        def perf(w): p_ret = (w@er); p_vol = np.sqrt(w.T@cov@w); return p_ret, p_vol
        def neg_sharpe(w): ret,vol = perf(w); return -(ret-0.045)/vol if vol>1e-8 else 1e9
        cons = ({'type':'eq','fun':lambda w: w.sum()-1})
        res = minimize(neg_sharpe, np.ones(n)/n, method='SLSQP', bounds=bnds, constraints=cons)
        if res.success:
            w = pd.Series(res.x, index=stocks); ret,vol = perf(w)
            return {'w':w,'er':ret,'vol':vol,'sharpe':(ret-0.045)/vol}
        else:
            w = pd.Series(1/n, index=stocks); return {'w':w,'er':er.mean(),'vol':np.sqrt(cov.mean().mean()),'sharpe':0.5}

class MonteCarlo:
    def __init__(self, opt, px, sims=25000, yrs=2):
        self.w, self.px, self.sims, self.days = opt['w'], px, sims, int(252*yrs)
    def run(self):
        print(f'🎲 Monte Carlo ({self.sims:,} sims)...')
        stks = self.w.index.intersection(self.px.columns); w = self.w[stks]/self.w[stks].sum()
        rets = self.px[stks].pct_change().dropna(); mu = rets.mean().values*252; cov = rets.cov().values*252
        L = np.linalg.cholesky(cov + np.eye(len(cov))*1e-8)
        dt = 1/252; paths = np.zeros((self.sims, self.days+1)); paths[:,0] = 100
        for s in range(self.sims):
            if s%5000==0 and s>0: print(f'  sim {s:,}/{self.sims:,}')
            shocks = np.random.normal(size=(self.days, len(stks))); corr = shocks @ L.T
            stk_vals = np.ones(len(stks))*100
            for t in range(self.days):
                daily = mu*dt + corr[t]*np.sqrt(dt); stk_vals *= (1+daily); paths[s,t+1] = (stk_vals@w)
        self.paths = paths
    def metrics(self):
        ret = self.paths[:,-1]/self.paths[:,0]-1
        return {'er':ret.mean(),'vol':ret.std(),'var5':np.percentile(ret,5),'var1':np.percentile(ret,1),
                'pprof':(ret>0).mean(),'ploss10':(ret<-0.1).mean(),'ploss20':(ret<-0.2).mean(),
                'best':ret.max(),'worst':ret.min(),'med':np.median(ret)}
    def plot(self, n=100, fn='mc.png'):
        plt.figure(figsize=(14,8)); t = np.arange(self.days+1)/252
        for i in range(min(n,self.sims)): plt.plot(t, self.paths[i,:], alpha=0.3, lw=0.5, color='blue')
        plt.plot(t, np.median(self.paths,axis=0), lw=3, color='red', label='Median')
        p5, p95 = np.percentile(self.paths,5,axis=0), np.percentile(self.paths,95,axis=0)
        plt.fill_between(t, p5, p95, alpha=0.2, color='gray', label='90% CI')
        plt.xlabel('Years'); plt.ylabel('Portfolio Value ($)'); plt.title(f'Monte Carlo ({self.sims:,} paths)'); plt.legend(); plt.grid(alpha=0.3)
        m=self.metrics(); plt.text(0.02,0.98,f"ER:{m['er']:.1%}  Vol:{m['vol']:.1%}  VaR5:{m['var5']:.1%}  P(>0):{m['pprof']:.1%}", transform=plt.gca().transAxes, va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        plt.tight_layout(); plt.savefig(fn, dpi=300); plt.show()

def run_all(tickers, initial_cash=500000, annual_withdrawal=10000, yrs=2, sims=25000, chart=True):
    print('='*80+'\n🚀 QUANTITATIVE WORKFLOW (WITHDRAWAL PLAN)\n'+'='*80)
    scr = FactorScreener(tickers); px, fd = scr.fetch_data()
    if px.empty or fd.empty: print('❌ data fail'); return None
    fct = scr.factors(); dcf = DCF(tickers, fd); proj = ReturnProj(tickers, scr, dcf); er = proj.proj()
    if er.empty: print('❌ proj fail'); return None
    opt = MeanVarOpt(er, px).opt(); mc = MonteCarlo(opt, px, sims=sims, yrs=yrs); mc.run(); m = mc.metrics()
    # withdrawal-adjusted ending value simulation
    withdraw_paths = mc.paths - np.arange(mc.paths.shape[1])*annual_withdrawal
    withdraw_final = withdraw_paths[:,-1]; w_ret = (withdraw_final/100)-1
    w_metrics = {'w_er':w_ret.mean(),'w_vol':w_ret.std(),'w_var5':np.percentile(w_ret,5),'w_pprof':(w_ret>0).mean()}
    if chart: mc.plot()
    # reports
    print('\n'+'─'*80+'\n🎯 SUMMARY (with $10k/yr withdrawal)\n'+'─'*80)
    print(pd.DataFrame({'Metric':['MVO ER','Vol','Sharpe','MC ER','MC Vol','5% VaR','P(>0)','W/Draw ER','W/Draw 5% VaR','W/Draw P(>0)'],
                        'Value':[f"{opt['er']:.2%}",f"{opt['vol']:.2%}",f"{opt['sharpe']:.2f}",f"{m['er']:.2%}",f"{m['vol']:.2%}",f"{m['var5']:.2%}",f"{m['pprof']:.2%}",f"{w_metrics['w_er']:.2%}",f"{w_metrics['w_var5']:.2%}",f"{w_metrics['w_pprof']:.2%}"]}).to_string(index=False))
    print('\n'+'─'*80+'\n💼 ALLOCATION ($500k start)\n'+'─'*80)
    alloc = pd.DataFrame({'Stock':opt['w'].index,'Weight':[f"{w:.2%}" for w in opt['w']],'StartSize':[f"${w*initial_cash:,.0f}" for w in opt['w']],'ER':[f"{er[s]:.2%}" for s in opt['w'].index]}).sort_values('Weight',ascending=False)
    print(alloc.to_string(index=False))
    print('\n'+'─'*80+'\n📈 STOCK DETAILS\n'+'─'*80)
    det = pd.DataFrame({'Ticker':er.index,'Price':[f"${fd.loc[s,'current_price']:.2f}" for s in er.index],'Score':[f"{fct.loc[s,'composite']:.3f}" if s in fct.index else "N/A" for s in er.index],'ER':[f"{er[s]:.2%}" for s in er.index],'DCF IV':[f"${dcf.details[s]['iv']:.2f}" if s in dcf.details else "N/A" for s in er.index],'MOS':[f"{dcf.details[s]['mos']:.1%}" if s in dcf.details else "N/A" for s in er.index]}).sort_values('ER',ascending=False)
    print(det.to_string(index=False))
    try: alloc.to_excel('alloc.xlsx'); det.to_excel('details.xlsx')
    except: pass
    return {'scr':scr,'dcf':dcf,'proj':proj,'opt':opt,'mc':mc,'m':m,'wm':w_metrics,'paths':mc.paths,'w_paths':withdraw_paths}

# ====================== RUN IN ONE CELL ===================================== #
tickers = ['AMZN','MSFT','GOOG','ANET','SNOW','CRWV','ASML','LRCX','NEE','NUE','ETN','VRT','TT','PWR','HUBB','PBW','ICLN','XYL','AWK','PLD','LEN','CTAS','ROK','KEYS','CSCO','WMT','HOOD','COST','IDXX','LIT','GLD']
results = run_all(tickers, initial_cash=500000, annual_withdrawal=10000, yrs=2, sims=25000, chart=True)
# ============================================================================= #

🚀 QUANTITATIVE WORKFLOW (WITHDRAWAL PLAN)


ValueError: Length of values (26) does not match length of index (30)